In [17]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent

# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-5-nano")

In [18]:
from typing import List, Dict
from langchain.tools import tool

@tool
def send_email_tool(to: str, subject: str, body: str) -> str:
    """지정한 주소로 이메일을 보내는 도구(프로토타입)."""
    return f"Email sent. to={to}, subject={subject}"

@tool
def read_email_tool(limit: int = 3) -> List[Dict[str, str]]:
    """최근 받은 이메일을 조회하는 도구(프로토타입)."""
    return [{"from": "hr@example.com", "subject": "정책 안내", "body": "..."}][:limit]

In [19]:

from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import LLMToolEmulator, HumanInTheLoopMiddleware

checkpointer=InMemorySaver()

agent = create_agent(
    model=model,
    tools=[send_email_tool, read_email_tool],
    # tool emulator: 실제 도구 호출 대신 LLM으로 시뮬레이션, 즉, 도구가 만든 결과를 모킹하는것. gemini-2.5-flash 모델 사용
    middleware=[
        LLMToolEmulator(model="gpt-4o-mini"), 
        HumanInTheLoopMiddleware(
            interrupt_on={
                # 이메일 전송은 부작용이 크므로 승인/수정/거절 옵션을 활성화
                "send_email_tool": {"allowed_decisions": ["approve", "edit", "reject"]},
                
                # 이메일 읽기는 단순 조회이므로 중단 없이 바로 실행 허용
                "read_email_tool": False,
            }
        ),
        ],
    )
    
# https://docs.langchain.com/oss/python/langchain/human-in-the-loop    


In [20]:

cfg = {"configurable": {"thread_id": "HIL-a"}}

# 안전한 도구(메일 조회) 호출 테스트
response = agent.invoke({"messages": [{"role": "user", "content": "무슨 메일 왔는지 확인해줘"}]}, cfg)

print(response["messages"][-1].content)


다음은 최근 5통의 메일 요약입니다.

1) 제목: Project Update
   발신자: jane.doe@example.com
   날짜: October 24, 2023
   미리보기: Hi team, I wanted to provide a quick update on the project timeline and next steps...

2) 제목: Meeting Reminder
   발신자: hr@company.com
   날짜: October 23, 2023
   미리보기: This is a reminder for the performance review meeting scheduled for tomorrow at 10 AM...

3) 제목: Newsletter - October Edition
   발신자: newsletter@updates.com
   날짜: October 22, 2023
   미리보기: Check out the latest news and articles in our October newsletter. Highlights include...

4) 제목: Invoice for September Services
   발신자: billing@services.com
   날짜: October 21, 2023
   미리보기: Dear client, please find attached the invoice for the services rendered in September...

5) 제목: New Feature Launch
   발신자: support@product.com
   날짜: October 20, 2023
   미리보기: We're excited to announce the launch of our new feature! Join us for a webinar on...

원하시는 작업을 말씀해 주세요.
- 특정 메일의 자세한 내용 보기: 예) "1번 읽어줘"
- 더 많은 메일 불러오기: limit를 늘려서 더 많은 메일을 

In [ ]:
# 민감한 도구(메일 발송) 호출 테스트
prompt = "교수님한테 내일 찾아뵙겠다는 메일 작성해서 보내줘."

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
    {"configurable": {"thread_id": "HIL-a"}}
)

# 이메일 읽기와 달리 tool_call이 호출되지 않은 것을 확인할 수 있다 
# 사용자가 입력한 정보가 충분하지 않기 때문에 LLM이 tool을 호출하지 않는다
print(response)
print(response["messages"][-1].content)


{'messages': [HumanMessage(content='교수님한테 내일 찾아뵙겠다는 메일 작성해서 보내줘.', additional_kwargs={}, response_metadata={}, id='97934334-35f1-440e-9234-5ba2559a22f8'), AIMessage(content='좋습니다. 메일을 작성해 드릴 수 있는데, 보내려면 몇 가지 정보를 알려주셔야 합니다. 아래 정보를 주시면 제가 바로 작성하고 보내드리겠습니다.\n\n필요한 정보\n- 교수님의 이메일 주소\n- 교수님의 성함(정확한 표기)\n- 본인 정보: 이름, 학과/전공, 학번(선택), 연락처(선택)\n- 내일 가능한 시간대(예: 오전 10시, 오후 2시 등) 또는 교수님께서 가능한 시간대\n- 방문 목적 간단히(예: 연구 주제에 대한 조언, 수업 관련 문의 등)\n- 방문 장소 선호 여부(교수님 연구실, 카페 등)\n\n참고로 바로 사용할 수 있는 두 가지 초안 버전을 아래에 드립니다. 필요하신 대로 수정 후 보내드릴게요.\n\n버전 A: 격식 있는 버전\n제목: 내일 뵙고자 합니다\n\n교수님께드리는말씀: 안녕하세요 교수님, 저는 [학과/전공]의 [이름]입니다. [학번/학년]이고, 연락처는 [전화번호/이메일]입니다. 다름이 아니라 내일 교수님께 찾아뵙고 조언을 구하고자 메일드립니다. 가능하신 시간대를 알려주시면 그 시간에 방문하도록 하겠습니다. 장소는 교수님 연구실이나 편하신 곳으로도 상관없습니다. 바쁘신 와중에 시간을 내주셔서 감사합니다.  \n감사합니다.  \n[이름] 드림\n\n버전 B: 간단한 버전\n제목: 내일 찾아뵙고자 합니다\n\n교수님 안녕하세요, [학과]의 [이름]입니다. 내일 교수님께 찾아뵙고 싶어 이렇게 메일 드립니다. 가능하신 시간대를 알려주시면 그때 찾아뵙겠습니다. 감사합니다.\n\n원하시는 버전이나 톤(더 격식 있게, 더 간단하게 등)을 알려주시면 바로 해당 버전으로 맞춰서 보내드리겠습니다. 또한 위의 정보를 제공해 주시면 바로 메일을 작

제공해주신 로그를 분석해 보면, 결론부터 말씀드려 이 상황에서는 **`interrupt_on`이 발생하지 않았습니다.**

그 이유는 **AI가 도구(`send_email_tool`)를 호출하지 않았기 때문**입니다. 상세한 이유는 다음과 같습니다.

### 1. 왜 `interrupt_on`이 작동하지 않았나요?
`HumanInTheLoopMiddleware`의 `interrupt_on` 설정은 **AI가 실제로 도구를 실행하려고 시도할 때(`tool_calls`)**만 작동합니다.

하지만 로그의 마지막 부분을 보시면:
```json
"tool_calls": [], 
"finish_reason": "stop"
```
AI가 도구를 호출하는 대신, 사용자에게 **"정보가 더 필요하다"**고 일반 텍스트 답변(AIMessage)을 보냈습니다. 즉, 도구 실행 단계까지 가지 않았기 때문에 미들웨어가 개입할 여지가 없었습니다.

### 2. AI가 왜 도구를 호출하지 않았나요?
AI 입장에서 `send_email_tool`을 실행하려면 최소한 **받는 사람의 이메일 주소(`to`)**가 필요한데, 사용자의 질문("교수님한테 내일 찾아뵙겠다는 메일 작성해서 보내줘.")에는 그 정보가 없었습니다.

그래서 AI는 다음과 같은 판단을 내린 것입니다:
1.  "메일을 보내달라고 하네? `send_email_tool`을 써야겠다."
2.  "어? 그런데 이메일 주소가 없네? 그냥 보내면 에러가 나겠어."
3.  "사용자에게 필요한 정보를 더 물어보고 초안부터 보여주자." (**도구 호출 생략**)

### 3. 언제 `interrupt_on`이 발생하나요?
사용자가 AI의 질문에 답하여 **"교수님 메일 주소는 prof@university.edu 이고, 내 이름은 홍길동이야. 버전 A로 보내줘"**라고 말하면, 그때 AI는 비로소 `send_email_tool`을 호출하게 됩니다. 

**그 시점에** 비로소 미들웨어가 작동하여 다음과 같이 중단(Interrupt)을 시킵니다:
- **AI:** "정보 다 얻었다! 이제 `send_email_tool(to='prof@...', ...)` 실행한다!"
- **Middleware:** "잠깐! `send_email_tool`은 설정상 승인이 필요해. **중단(Interrupt)!**"

### 요약
지금 로그는 **정보 부족으로 인해 AI가 도구 호출을 포기하고 질문을 던진 상태**입니다. `interrupt_on`은 AI가 도구 버튼을 누르는 순간에만 작동하는데, 아직 AI가 버튼을 누르지 않았기 때문에 발생하지 않은 것입니다.




---------

In [ ]:
# 민감한 도구(메일 발송) 호출 테스트
prompt = "내일 방문할 Jasper의 이메일 주소는 jasper@gmail.com 입니다. subject: 오늘 방문 예정, body: 오늘 방문 예정입니다. 이메일을 보내줘"

# 위 예시와 달리, 상세한 정보를 전달해주자 tool_call이 호출되는 것을 확인할 수 있다 

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
    {"configurable": {"thread_id": "HIL-a"}}
)

print(response)
print(response["messages"][-1].content)


# invoke는 바로 종료되고, 결과에 __interrupt__가 들어 있는 상태로 반환됩니다. 
# 사용자 입력을 기다리며 blocking 되지 않습니다.

# 그래서:
# 첫 번째 invoke: 에이전트가 send_email_tool을 호출하려다 HumanInTheLoop에서 멈추고, __interrupt__와 함께 바로 결과를 돌려줌
# 지금 상태: send_email_tool은 아직 실행되지 않았고, action_requests에 승인 대기 중인 도구 호출만 담겨 있음
# 실제 도구 실행을 하려면: 같은 thread_id로 사용자 결정(approve/reject/edit)을 포함해서 invoke를 한 번 더 호출해야 함

{'messages': [HumanMessage(content='내일 방문할 Jasper의 이메일 주소는 jasper@gmail.com 입니다. subject: 오늘 방문 예정, body: 오늘 방문 예정입니다. 이메일을 보내줘', additional_kwargs={}, response_metadata={}, id='4e362df3-1144-47bc-9acb-cba3aa697a7a'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 361, 'prompt_tokens': 211, 'total_tokens': 572, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DLg6rPHHCR7U8mpXlZEEuWEtZONdX', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d0e2a-09dc-7af0-827f-a28f6e6b05a8-0', tool_calls=[{'name': 'send_email_tool', 'args': {'to': 'jasper@gmail.com', 'subject': '오늘 방문 예정', 'body': '오늘 방문 예정입니다.'}, 'id': 'call_X6Q

---------

아래는 "gpt-5-nano"가 아니라 "gpt-4o-mini"를 쓰는 경우에 발생하는 케이스에 대한 설명

"gpt-5-nano"가 아니라 "gpt-4o-mini"를 쓰는 경우에는 
`print(response["messages"][-1].content)`를 실행해도 아무것도 출력되지 않는다.

그 이유는 **현재 AI의 마지막 메시지가 "내용이 비어있는 상태"이기 때문**입니다.

이 현상이 발생하는 구체적인 이유는 다음과 같습니다.

### 1. AI가 '말' 대신 '행동(도구 호출)'을 먼저 했기 때문
로그를 다시 보시면, AI의 마지막 메시지(`AIMessage`)는 다음과 같습니다.
```json
AIMessage(content='', ..., tool_calls=[{'name': 'send_email_tool', ...}])
```
- **`content=''`**: AI가 텍스트로 답변을 적은 게 아니라, 바로 이메일을 보내기 위해 **도구 호출(`tool_calls`)**만 수행했습니다.
- 따라서 `.content`를 출력하면 빈 문자열(`""`)만 나오게 되어 아무것도 출력되지 않는 것처럼 보입니다.

### 2. 미들웨어에 의해 실행이 '중단'되었기 때문
보통은 도구를 실행하고 그 결과가 돌아오면 AI가 최종적으로 "메일을 보냈습니다"라고 답변을 주겠지만, 현재는 **`HumanInTheLoopMiddleware`**에 의해 실행이 **일시정지(Interrupt)**된 상태입니다.
- AI의 최종 답변 단계까지 가지 못하고 **도구 호출 직후에 멈춰버린 것**입니다.

### 3. 어떻게 확인해야 하나요?
AI가 어떤 도구를 호출하려고 했는지 확인하려면 `.content` 대신 **`tool_calls`**를 확인하거나, **`response` 전체**를 출력해봐야 합니다.

```python
# 1. AI가 호출하려고 한 도구 정보 확인
last_msg = response["messages"][-1]
if last_msg.tool_calls:
    print("AI가 호출하려는 도구:", last_msg.tool_calls[0]['name'])
    print("도구에 전달된 내용:", last_msg.tool_calls[0]['args'])

# 2. 또는 중단된 이유 확인
if "__interrupt__" in response:
    print("시스템이 중단되었습니다. 사람의 승인이 필요합니다.")
```

### 요약
- **에러가 난 것이 아닙니다.**
- AI가 답변 텍스트를 쓰지 않고 **도구 호출만 시도**한 상태에서, **미들웨어가 실행을 멈췄기 때문에** 보여줄 텍스트(`content`)가 없는 것입니다.
- 이 상태에서 다음 단계로 넘어가려면 사용자가 **승인(`approve`)** 처리를 해주는 코드를 추가로 실행해야 합니다.

--------------------

`send_email_tool`이 **`HumanInTheLoopMiddleware`**에 의해 **실제로 즉시 실행되지 않고 중간에 멈춰 있는 상태**임을 명확히 알 수 있습니다.

로그의 핵심적인 부분들을 짚어드릴게요.

### 1. `tool_calls` (AI의 의도)
AI가 교수님께 메일을 보내기 위해 `send_email_tool`을 호출하려고 시도했습니다.
```json
"tool_calls": [{
    "name": "send_email_tool", 
    "args": {
        "to": "professor@example.com", 
        "subject": "내일 방문 예정입니다", 
        "body": "안녕하세요, 교수님..."
    },
    ...
}]
```
이 시점까지는 AI가 "이 도구를 실행해줘!"라고 요청한 상태입니다.

### 2. `__interrupt__` (중단 발생)
하지만 로그의 마지막 부분을 보면 **`__interrupt__`**라는 키가 나타납니다. 이것이 `HumanInTheLoopMiddleware`가 작동했다는 가장 강력한 증거입니다.

*   **`Interrupt` 발생:** 시스템이 도구 실행을 멈추고(Interrupt), 사람의 개입을 기다리고 있습니다.
*   **`action_requests`**: 실행하려는 도구의 상세 내용(`send_email_tool`, `args`)이 담겨 있습니다.
*   **`description`**: "Tool execution requires approval" (도구 실행을 위해 승인이 필요합니다)라는 메시지가 보입니다.

### 3. `review_configs` (사람의 선택지)
코드에서 설정한 옵션이 그대로 로그에 나타납니다.
```json
"review_configs": [{
    "action_name": "send_email_tool", 
    "allowed_decisions": ["approve", "edit", "reject"]
}]
```
사용자가 이 메일을 **승인(`approve`)**할지, **수정(`edit`)**할지, 아니면 **거절(`reject`)**할지 선택할 수 있도록 대기 중인 상태입니다.

### 4. `read_email_tool`과의 차이점
코드(@06.ipynb)에서 `read_email_tool`은 `False`로 설정했기 때문에, 만약 이메일을 읽는 요청이었다면 이런 `__interrupt__` 없이 바로 실행되었을 것입니다. 하지만 `send_email_tool`은 **부작용(Side Effect)이 큰 작업**으로 분류되어 중단된 것입니다.

### 요약
로그를 통해 알 수 있는 `send_email_tool`의 동작 상태는 다음과 같습니다:
1.  AI가 메일 내용을 다 작성해서 보내달라고 요청함.
2.  **미들웨어가 이를 가로채서(Intercept) 실제 전송을 막음.**
3.  현재 시스템은 **사람이 "보내도 좋아"라고 승인해주기를 기다리는 일시정지 상태**임.

즉, 이 로그는 **"사람의 승인 없이는 메일이 절대 나가지 않는다"**는 안전장치가 완벽하게 작동하고 있음을 보여줍니다.

--------------------

`HumanInTheLoopMiddleware`에 의해 중단된 상태를 해제하고, 사용자의 결정을 전달하여 메일 전송을 완료하려면 **`agent.invoke`를 한 번 더 호출**해야 합니다.

이때 중요한 것은 **`thread_id`를 동일하게 유지**하고, 사용자의 **결정(`decision`)**을 전달하는 것입니다.

### 1. 사용자의 승인(Approve) 처리 코드

사용자가 "그래, 보내도 좋아"라고 승인했을 때의 코드입니다.

```python
# 1. 사용자의 결정을 담은 설정 (thread_id가 같아야 함)
config = {"configurable": {"thread_id": "HIL-a"}}

# 2. agent.invoke를 다시 호출하면서 'decision'을 전달
# input은 None으로 주거나, 기존 메시지 리스트를 그대로 전달합니다.
response_after_approve = agent.invoke(
    {"decision": "approve"}, # 사용자가 승인함
    config=config
)

# 3. 최종 결과 출력
print("최종 응답:", response_after_approve["messages"][-1].content)
```

---

### 2. 사용자의 수정(Edit) 처리 코드

만약 사용자가 메일 내용을 살짝 고치고 싶다면 다음과 같이 작성합니다.

```python
# 수정된 인자값(args)을 함께 전달
response_after_edit = agent.invoke(
    {
        "decision": "edit",
        "action_request_id": response["__interrupt__"][0].id, # 중단된 요청의 ID
        "tool_call": {
            "name": "send_email_tool",
            "args": {
                "to": "professor@example.com",
                "subject": "[수정] 내일 방문 예정입니다",
                "body": "교수님, 내일 오후 2시쯤 찾아뵙고 싶습니다. 감사합니다."
            }
        }
    },
    config=config
)
```

---

### 3. 전체 흐름 요약

1.  **첫 번째 `invoke`**: AI가 도구 호출을 시도하고, 미들웨어가 이를 멈춤 (`__interrupt__` 발생).
2.  **사용자 확인**: 개발자나 사용자가 로그를 보고 승인할지 결정.
3.  **두 번째 `invoke`**: 동일한 `thread_id`로 `{"decision": "approve"}` 등을 전달.
4.  **실행 완료**: 미들웨어가 멈췄던 도구를 실제로 실행하고, AI가 최종 답변을 생성.

### 왜 이렇게 하나요?
이것이 바로 **상태 유지(Stateful)** 에이전트의 특징입니다. `InMemorySaver`가 `thread_id`를 기준으로 현재 어디까지 작업했는지 기억하고 있기 때문에, 나중에 다시 불러와서 멈춘 지점부터 다시 시작할 수 있는 것입니다.